# Sawti-ASR — lanceur Kaggle

Notebook poussé par `kaggle kernels push` depuis `kaggle/` du dépôt.
Il charge les secrets, clone le dépôt, installe, lance les tests, puis exécute la cellule **COMMANDES**.
Aucun jeton n'est écrit ici : ils sont lus dans le dataset Kaggle PRIVÉ `sawti-secrets`.

## COMMANDES (la seule cellule à modifier)

In [ ]:
# Une commande shell par ligne, lancées dans l'ordre depuis la racine du dépôt.
# Le notebook s'arrête à la première commande qui échoue.
COMMANDES = [
    "python scripts/prepare_data.py --no-moulsot --no-casablanca",
]

REPO_URL = "https://github.com/Mohamedaminebelasri/sawti-asr"
BRANCHE = "main"

## 1. Jetons (dataset Kaggle privé sawti-secrets)

In [ ]:
import glob, os

# Les jetons viennent du dataset PRIVÉ sawti-secrets (fichier secrets.env, lignes NOM=valeur),
# attaché par dataset_sources dans kernel-metadata.json. On n'affiche JAMAIS une valeur.
SECRETS = ["HF_TOKEN", "MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]
fichiers = sorted(glob.glob("/kaggle/input/**/secrets.env", recursive=True))
print("Fichier de jetons trouvé :", "oui" if fichiers else "non")
lus = {}
if fichiers:
    with open(fichiers[0], encoding="utf-8-sig") as f:
        for ligne in f:
            ligne = ligne.strip()
            if not ligne or ligne.startswith("#") or "=" not in ligne:
                continue
            nom, valeur = ligne.split("=", 1)
            lus[nom.strip()] = valeur.strip().strip('"').strip("'")
for nom in SECRETS:
    if lus.get(nom):
        os.environ[nom] = lus[nom]
        print(f"SECRET {nom}: OK")
    else:
        print(f"SECRET {nom}: ABSENT")
del lus
if os.environ.get("HF_TOKEN"):
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

## 2. Dépôt, dépendances, ESC-50

In [ ]:
import subprocess, sys, time

TRAVAIL = "/kaggle/temp/sawti-asr" if os.path.isdir("/kaggle") else os.path.abspath("sawti-asr")
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("sortie")
os.makedirs(SORTIE, exist_ok=True)
JOURNAL = os.path.join(SORTIE, "run_log.txt")

def sh(cmd, cwd=None):
    """Lance une commande, affiche et journalise sa sortie, s'arrête si elle échoue."""
    debut = time.time()
    print(f"\n$ {cmd}", flush=True)
    with open(JOURNAL, "a", encoding="utf-8") as j:
        j.write(f"\n$ {cmd}\n")
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="", flush=True)
            j.write(ligne)
        p.wait()
        duree = time.time() - debut
        j.write(f"[code retour {p.returncode}, {duree:.0f} s]\n")
    print(f"[code retour {p.returncode}, {duree:.0f} s]", flush=True)
    if p.returncode != 0:
        raise RuntimeError(f"Échec ({p.returncode}) : {cmd}")

sh(f"mkdir -p {os.path.dirname(TRAVAIL)} && rm -rf {TRAVAIL} && git clone -q --depth 1 -b {BRANCHE} {REPO_URL} {TRAVAIL}")
sh("git log -1 --format='commit %h %s'", cwd=TRAVAIL)
sh("nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'pas de GPU'")
sh(f"{sys.executable} -m pip install -q -r requirements.txt", cwd=TRAVAIL)
sh(f"{sys.executable} -m pip install -q -e .", cwd=TRAVAIL)
sh("git clone -q --depth 1 https://github.com/karolpiczak/ESC-50 data/ESC-50", cwd=TRAVAIL)

## 2 bis. Données : rechargement depuis le Hub + jeux bruités

Si `data/processed` n'existe pas, il est rechargé depuis le dataset PRIVÉ du Hub (`hub.repo_id`), puis les jeux bruités sont reconstruits par `build_test_sets.py`. Sans dataset sur le Hub, rien n'est fait.

In [ ]:
sh(f"{sys.executable} scripts/fetch_data.py", cwd=TRAVAIL)

## 3. Tests

In [ ]:
sh(f"{sys.executable} -m pytest -q", cwd=TRAVAIL)

## 4. Exécution des COMMANDES

In [ ]:
for c in COMMANDES:
    sh(c, cwd=TRAVAIL)

# Copie des petits résultats dans la sortie du notebook (pas les données).
import shutil
res = os.path.join(TRAVAIL, "results")
if os.path.isdir(res):
    shutil.copytree(res, os.path.join(SORTIE, "results"), dirs_exist_ok=True)
print("\nTERMINE : toutes les commandes ont réussi.")